<a href="https://colab.research.google.com/github/Not-kh-lily-23/dbank-longitudinal-prediction/blob/main/cohort_tracking.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [4]:
import pandas as pd
import numpy as np
import os
from google.colab import drive
drive.mount('/content/drive')
drive='/content/drive/MyDrive/DementiaBank Project'
df=pd.read_csv(os.path.join(drive,'pitt_corpus_content.csv'))
df[['participant_id','visit_number']]=df['file_id'].str.extract(r'([A-Za-z0-9]+)-?(\d*)')
df['visit_number']=df['visit_number'].replace('','0').astype(int)
df['group']=df['group'].astype(str).str.strip().str.lower()
print(f"Unique groups found in your data: {df['group'].unique()}")
df=df.sort_values(by=['participant_id','visit_number'])
cohorts=[]
for participant, group_df in df.groupby('participant_id'):
    first_diagnosis=group_df.iloc[0]['group']
    last_diagnosis=group_df.iloc[-1]['group']
    num_visits=len(group_df)
    status="Unknown/Excluded"
    if 'control' in first_diagnosis and 'control' in last_diagnosis:
        status="Stable Control"
    elif ('control' in first_diagnosis or 'mci' in first_diagnosis) and ('dementia' in last_diagnosis or 'ad' in last_diagnosis):
        status="Converter"
    elif ('dementia' in first_diagnosis or 'ad' in first_diagnosis):
        status="Stable AD"
    cohorts.append({
        'participant_id': participant,
        'baseline_diagnosis': first_diagnosis,
        'final_diagnosis': last_diagnosis,
        'total_visits': num_visits,
        'cohort_status': status
    })
cohort_df=pd.DataFrame(cohorts)
final_df=pd.merge(df,cohort_df[['participant_id','cohort_status']],on='participant_id',how='left')
final_df.to_csv(os.path.join(drive,'pitt_corpus_longitudinal_master.csv'),index=False)
print("Cohort Counts")
counts=cohort_df['cohort_status'].value_counts()
print(counts)
converter_N = counts.get('Converter', 0)
print(f"\nTotal True Converters: {converter_N}")
if converter_N >= 15:
    print("Passed: Proceed with Longitudinal Predictor Hypothesis.")
else:
    print("Failed: Pivot to cross-sectional severity stratification.")

Mounted at /content/drive
Unique groups found in your data: ['nan' 'probablead' 'mci' 'possiblead' 'memory' 'vascular' 'control']
Cohort Counts
cohort_status
Unknown/Excluded    255
Stable AD            25
Stable Control       12
Name: count, dtype: int64

Total True Converters: 0
Failed: Pivot to cross-sectional severity stratification.
